In [ ]:
import json
import os
DATA_FILE = "students.json"

class Student:
    def __init__(self, student_id, name, age, course, semester):
        self.student_id = student_id
        self.name = name
        self.age = age
        self.course = course
        self.semester = semester
        self.subjects = {}
    def add_subject(self, subject, marks):
        self.subjects[subject] = marks
    def calculate_total(self):
        if len(self.subjects) == 0:
            return 0
        total = 0
        for marks in self.subjects.values():
            total = total + marks
        return total
    def calculate_percentage(self):
        if len(self.subjects) == 0:
            return 0
        total = self.calculate_total()
        maximum_marks = len(self.subjects) * 100
        percentage = (total / maximum_marks) * 100
        return percentage
    def calculate_average(self):
        if len(self.subjects) == 0:
            return 0
        total = self.calculate_total()
        return total / len(self.subjects)
    def calculate_gpa(self):
        if len(self.subjects) == 0:
            return 0
        percentage = self.calculate_percentage()

        if percentage >= 90:
            return 10.0
        elif percentage >= 80:
            return 9.0
        elif percentage >= 70:
            return 8.0
        elif percentage >= 60:
            return 7.0
        elif percentage >= 50:
            return 6.0
        elif percentage >= 40:
            return 5.0
        else:
            return 0.0

    def get_grade(self):
        percentage = self.calculate_percentage()

        if percentage >= 90:
            return "A+"
        elif percentage >= 80:
            return "A"
        elif percentage >= 70:
            return "B+"
        elif percentage >= 60:
            return "B"
        elif percentage >= 50:
            return "C"
        elif percentage >= 40:
            return "D"
        else:
            return "F"

    def get_status(self):
        if len(self.subjects) == 0:
            return "No marks available"

        for marks in self.subjects.values():
            if marks < 40:
                return "Fail"

        return "Pass"
    def get_highest_subject(self):
        if len(self.subjects) == 0:
            return "No subjects"
        highest_subject = max(self.subjects, key=self.subjects.get)
        return highest_subject
        
    def get_lowest_subject(self):
        if len(self.subjects) == 0:
            return "No subjects"
        lowest_subject = min(self.subjects, key=self.subjects.get)
        return lowest_subject

    def to_dictionary(self):
        return {
            "student_id": self.student_id,
            "name": self.name,
            "age": self.age,
            "course": self.course,
            "semester": self.semester,
            "subjects": self.subjects
        }

    def display(self):
        print("\n==========================================")
        print("             STUDENT DETAILS")
        print("==========================================")

        print("Student ID :", self.student_id)
        print("Name       :", self.name)
        print("Age        :", self.age)
        print("Course     :", self.course)
        print("Semester   :", self.semester)

        print("\n---------- SUBJECT MARKS ----------")

        if len(self.subjects) == 0:
            print("No subjects added.")
        else:
            for subject, marks in self.subjects.items():
                print(subject, ":", marks)

        print("-----------------------------------")

        print("Total      :", self.calculate_total())
        print("Percentage :", round(self.calculate_percentage(), 2), "%")
        print("Average    :", round(self.calculate_average(), 2))
        print("GPA        :", self.calculate_gpa())
        print("Grade      :", self.get_grade())
        print("Status     :", self.get_status())

        if len(self.subjects) > 0:
            print("Highest    :", self.get_highest_subject())
            print("Lowest     :", self.get_lowest_subject())

        print("==========================================")

    @staticmethod
    def from_dictionary(data):
        student = Student(
            data["student_id"],
            data["name"],
            data["age"],
            data["course"],
            data["semester"]
        )

        student.subjects = data.get("subjects", {})
        return student

class StudentManager:
    def __init__(self):
        self.students = []
        self.load_students()
    def load_students(self):
        if os.path.exists(DATA_FILE):
            try:
                f = open(DATA_FILE, "r")
                data = json.load(f)
                f.close()
                self.students = []

                for item in data:
                    student = Student.from_dictionary(item)
                    self.students.append(student)

            except json.JSONDecodeError:
                print("The student data file is corrupted.")
                self.students = []
        else:
            self.students = []

    def save_students(self):
        data = []
        for student in self.students:
            data.append(student.to_dictionary())
            
        f = open(DATA_FILE, "w")
        json.dump(data, f, indent=4)
        f.close()

    def find_by_id(self, student_id):
        for student in self.students:
            if student.student_id == student_id:
                return student
        return None

    def add_student(self):

        print("\n========== ADD NEW STUDENT ==========")
        student_id = input("Enter student ID: ")
        if self.find_by_id(student_id) is not None:
            print("A student with this ID already exists.")
            return

        name = input("Enter student name: ")
        age = self.get_integer("Enter age: ", 1, 100)
        course = input("Enter course: ")
        semester = self.get_integer("Enter semester: ", 1, 12)
        
        student = Student(
            student_id,
            name,
            age,
            course,
            semester
        )

        print("\nNow enter subject information.")
        print("Enter 'done' when you have finished.")

        while True:
            subject = input("Enter subject name: ")

            if subject.lower() == "done":
                break

            if subject == "":
                print("Subject name cannot be empty.")
                continue

            if subject in student.subjects:
                print("This subject already exists.")
                continue

            marks = self.get_integer(
                "Enter marks out of 100: ",
                0,
                100
            )

            student.add_subject(subject, marks)
        
        self.students.append(student)
        self.save_students()

        print("\nStudent added successfully!")

    def display_all(self):
        print("\n========== ALL STUDENTS ==========")

        if len(self.students) == 0:
            print("No student records found.")
            return

        for student in self.students:

            print(
                student.student_id,
                "|",
                student.name,
                "|",
                student.course,
                "|",
                round(student.calculate_percentage(), 2),
                "%",
                "|",
                student.get_grade()
            )

        print("==================================")

    def view_student(self):
        student_id = input("Enter student ID: ")
        student = self.find_by_id(student_id)
        if student is None:
            print("Student not found.")
        else:
            student.display()

    def search_student(self):
        print("\n========== SEARCH STUDENT ==========")
        print("1. Search by ID")
        print("2. Search by name")

        choice = input("Enter choice: ")

        if choice == "1":
            student_id = input("Enter student ID: ")
            student = self.find_by_id(student_id)

            if student is None:
                print("Student not found.")
            else:
                student.display()

        elif choice == "2":
            name = input("Enter student name: ")
            found = False

            for student in self.students:
                if name.lower() in student.name.lower():
                    print(
                        student.student_id,
                        "|",
                        student.name,
                        "|",
                        student.course,
                        "|",
                        student.get_grade()
                    )

                    found = True

            if found == False:
                print("No students found.")

        else:
            print("Invalid choice.")

    def update_student(self):
        student_id = input("Enter student ID to update: ")
        student = self.find_by_id(student_id)

        if student is None:
            print("Student not found.")
            return

        while True:
            print("\n========== UPDATE STUDENT ==========")
            print("1. Update name")
            print("2. Update age")
            print("3. Update course")
            print("4. Update semester")
            print("5. Add/update subject marks")
            print("6. Delete subject")
            print("7. Finish update")

            choice = input("Enter choice: ")

            if choice == "1":
                student.name = input("Enter new name: ")

            elif choice == "2":
                student.age = self.get_integer(
                    "Enter new age: ",
                    1,
                    100
                )

            elif choice == "3":
                student.course = input("Enter new course: ")

            elif choice == "4":
                student.semester = self.get_integer(
                    "Enter new semester: ",
                    1,
                    12
                )

            elif choice == "5":
                subject = input("Enter subject name: ")

                marks = self.get_integer(
                    "Enter marks out of 100: ",
                    0,
                    100
                )

                student.add_subject(subject, marks)
                print("Subject marks updated.")

            elif choice == "6":
                subject = input("Enter subject to delete: ")

                if subject in student.subjects:
                    del student.subjects[subject]
                    print("Subject deleted.")
                else:
                    print("Subject not found.")

            elif choice == "7":
                self.save_students()

                print("Student updated successfully.")
                break

            else:
                print("Invalid choice.")

    def delete_student(self):
        student_id = input("Enter student ID to delete: ")
        student = self.find_by_id(student_id)

        if student is None:
            print("Student not found.")
            return

        print("\nStudent found:")
        print(student.name)

        confirm = input(
            "Are you sure you want to delete this student? (y/n): "
        )

        if confirm.lower() == "y":
            self.students.remove(student)
            self.save_students()
            print("Student deleted successfully.")

        else:
            print("Deletion cancelled.")

    def class_report(self):
        print("\n========== CLASS REPORT ==========")

        if len(self.students) == 0:
            print("No student data available.")
            return

        total_percentage = 0
        passed = 0
        failed = 0

        grade_count = {
            "A+": 0,
            "A": 0,
            "B+": 0,
            "B": 0,
            "C": 0,
            "D": 0,
            "F": 0
        }

        top_student = self.students[0]
        
        for student in self.students:
            percentage = student.calculate_percentage()
            total_percentage = total_percentage + percentage

            if student.get_status() == "Pass":
                passed = passed + 1
            elif student.get_status() == "Fail":
                failed = failed + 1

            grade = student.get_grade()

            if grade in grade_count:
                grade_count[grade] = grade_count[grade] + 1

            if percentage > top_student.calculate_percentage():
                top_student = student

        class_average = total_percentage / len(self.students)

        print("Total students :", len(self.students))
        print("Class average  :", round(class_average, 2), "%")
        print("Passed         :", passed)
        print("Failed         :", failed)

        print("\nGrade Distribution")

        for grade, count in grade_count.items():
            print(grade, ":", count)

        print("\nTop Performer")
        print("ID         :", top_student.student_id)
        print("Name       :", top_student.name)
        print(
            "Percentage :",
            round(top_student.calculate_percentage(), 2),
            "%"
        )
        print("Grade      :", top_student.get_grade())

        print("==================================")

    def subject_report(self):
        print("\n========== SUBJECT REPORT ==========")

        if len(self.students) == 0:
            print("No student data available.")
            return

        subject_data = {}

        for student in self.students:

            for subject, marks in student.subjects.items():
                if subject not in subject_data:
                    subject_data[subject] = []

                subject_data[subject].append(marks)

        if len(subject_data) == 0:
            print("No subject marks available.")
            return

        for subject, marks_list in subject_data.items():
            total = sum(marks_list)
            average = total / len(marks_list)

            highest = max(marks_list)
            lowest = min(marks_list)

            print("\nSubject:", subject)
            print("Students :", len(marks_list))
            print("Average  :", round(average, 2))
            print("Highest  :", highest)
            print("Lowest   :", lowest)

        print("====================================")

    def student_report(self):
        student_id = input("Enter student ID: ")
        student = self.find_by_id(student_id)

        if student is None:
            print("Student not found.")
            return

        print("\n========== STUDENT REPORT ==========")

        print("Student ID :", student.student_id)
        print("Name       :", student.name)
        print("Course     :", student.course)
        print("Semester   :", student.semester)

        print("\nAcademic Performance")

        for subject, marks in student.subjects.items():

            if marks >= 90:
                grade = "A+"
            elif marks >= 80:
                grade = "A"
            elif marks >= 70:
                grade = "B+"
            elif marks >= 60:
                grade = "B"
            elif marks >= 50:
                grade = "C"
            elif marks >= 40:
                grade = "D"
            else:
                grade = "F"

            print(
                subject,
                ":",
                marks,
                "/ 100",
                "| Grade:",
                grade
            )

        print("\nOverall Percentage:",
              round(student.calculate_percentage(), 2), "%")

        print("GPA:", student.calculate_gpa())
        print("Overall Grade:", student.get_grade())
        print("Result:", student.get_status())

        if len(student.subjects) > 0:
            print(
                "Best Subject:",
                student.get_highest_subject()
            )
            print(
                "Weakest Subject:",
                student.get_lowest_subject()
            )
        print("====================================")

    @staticmethod
    def get_integer(message, minimum, maximum):
        while True:
            try:
                value = int(input(message))

                if value >= minimum and value <= maximum:
                    return value

                print(
                    "Enter a value between",
                    minimum,
                    "and",
                    maximum
                )

            except ValueError:
                print("Please enter a valid number.")

def main():
    manager = StudentManager()
    while True:
        print("\n")
        print("==============================================")
        print("        STUDENT MANAGEMENT SYSTEM")
        print("==============================================")
        print("1. Add new student")
        print("2. Display all students")
        print("3. View student profile")
        print("4. Search student")
        print("5. Update student")
        print("6. Delete student")
        print("7. Student report")
        print("8. Class report")
        print("9. Subject-wise report")
        print("10. Exit")
        print("==============================================")
        
        choice = input("Enter your choice: ")

        if choice == "1":
            manager.add_student()

        elif choice == "2":
            manager.display_all()

        elif choice == "3":
            manager.view_student()

        elif choice == "4":
            manager.search_student()

        elif choice == "5":
            manager.update_student()

        elif choice == "6":
            manager.delete_student()

        elif choice == "7":
            manager.student_report()

        elif choice == "8":
            manager.class_report()

        elif choice == "9":
            manager.subject_report()

        elif choice == "10":
            manager.save_students()

            print("\nStudent data saved.")
            print("Thank you for using the system!")
            break

        else:
            print("Invalid choice. Please try again.")

main()



        STUDENT MANAGEMENT SYSTEM
1. Add new student
2. Display all students
3. View student profile
4. Search student
5. Update student
6. Delete student
7. Student report
8. Class report
9. Subject-wise report
10. Exit


Enter your choice:  1



========== ADD NEW STUDENT ==========


Enter student ID:  19872022
Enter student name:  LionelMessi
Enter age:  19
Enter course:  AcademiaFootBall
Enter semester:  3



Now enter subject information.
Enter 'done' when you have finished.


...

Student updated successfully.


        STUDENT MANAGEMENT SYSTEM
1. Add new student
2. Display all students
3. View student profile
4. Search student
5. Update student
6. Delete student
7. Student report
8. Class report
9. Subject-wise report
10. Exit


Enter your choice:  8



========== CLASS REPORT ==========
Total students : 3
Class average  : 94.6 %
Passed         : 3
Failed         : 0

Grade Distribution
A+ : 3
A : 0
B+ : 0
B : 0
C : 0
D : 0
F : 0

Top Performer
ID         : 19872022
Name       : LionelMessi
Percentage : 100.0 %
Grade      : A+


        STUDENT MANAGEMENT SYSTEM
1. Add new student
2. Display all students
3. View student profile
4. Search student
5. Update student
6. Delete student
7. Student report
8. Class report
9. Subject-wise report
10. Exit


Enter your choice:  9



========== SUBJECT REPORT ==========

Subject: Dribbling
Students : 3
Average  : 94.67
Highest  : 100
Lowest   : 84

Subject: Tackling
Students : 3
Average  : 97.0
Highest  : 100
Lowest   : 91

Subject: BreakingDefense
Students : 3
Average  : 96.0
Highest  : 100
Lowest   : 93

Subject: Passing
Students : 3
Average  : 87.67
Highest  : 100
Lowest   : 73

Subject: Striking 
Students : 3
Average  : 97.67
Highest  : 100
Lowest   : 93


        STUDENT MANAGEMENT SYSTEM
1. Add new student
2. Display all students
3. View student profile
4. Search student
5. Update student
6. Delete student
7. Student report
8. Class report
9. Subject-wise report
10. Exit
